<a href="https://colab.research.google.com/github/NicolasHillingdale/Part1-Basic_Agent_Part_1_Colab.ipynb/blob/test/Part1/Basic_Agent_Part_1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Last Updated: August 2026
(Tested on Google Colab)

In [19]:
!python --version

Python 3.12.13


In [30]:
!pip install -q -U "tavily-python>=0.7.0" "crewai[tools]" "python-dotenv" "langchain-anthropic" "pandas" "langchain_community"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.9/753.9 kB 38.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.20 requires langchain-community<0.1,>=0.0.38, but you have langchain-community 0.4.2 which is incompatible.
langchain 0.1.20 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.6.1 which is incompatible.
langchain 0.1.20 requires langchain-

In [ ]:
# Restart code
import os
os.kill(os.getpid(), 9)

In [6]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [7]:
import crewai
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from tavily import TavilyClient
import json

In [8]:
client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool("Tavily Search")
def tavily_search(query: str) -> str:
  """Search the web for recent renewable energy deal news. Returns JSON results with URLs."""
  r = client.search(
        query=query,
        search_depth="advanced",
        topic="news",
        time_range="week",
        max_results=10,
        include_raw_content=True,
    )
  return json.dumps(r["results"], indent=2)


@tool("Tavily Extract")
def tavily_extract(urls: str) -> str:
    """Extract full page content from one or more comma-separated URLs."""
    return json.dumps(client.extract([u.strip() for u in urls.split(",")]), indent=2)

In [10]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Opportunity(BaseModel):
  company_name: str
  country: str
  region: str
  technology: str
  deal_type: str
  stage: str
  signal: str
  deal_size: Optional[str] = None
  source_url: str
  date_announced: Optional[str] = None

class OpportunityList(BaseModel):
  opportunities: List[Opportunity]

class ScoredOpportunity(Opportunity):
  mandate_fit_score: int = Field(ge=1, le=10)
  financing_need_score: int = Field(ge=1, le=10)
  engagement_likelihood_score: int = Field(ge=1, le=10)
  overall_score: float
  rationale: str
  suggested_role: str              # Advisor / Equity / Debt / Monitor

class ScoredList(BaseModel):
  scored: List[ScoredOpportunity]

In [8]:
researcher = Agent(
    role="researcher",
    goal="Build a list of renewable energy companies in Africa and Europe renewable energy market looking for project finance.",
    backstory="You are a research analyst at Hillingdale Capital, a specialist infrastructure and renewables investment firm. Your task is to research and compile a tracker of recent deal activity in the African and European renewable energy markets, covering IPPs (Independent Power Producers), project developers, and utility-scale projects. The purpose is to identify opportunities for Hillingdale Capital to approach and connect with relevant parties.",
    verbose=True
)

In [9]:
evaluator = Agent(
    role="evaluator",
    goal="Evaluate a list of companies in the renewable energy sector and derive a score between 1 - 10, 1 being they require no financing and 10 being they really do need project or growth finance from the {companies_list}",
    backstory="You are a evaluator analyst at Hillingdale Capital, a specialist infrastructure and renewables investment firm. Your task is to research and compile a tracker of recent deal activity in the African and European renewable energy markets, covering IPPs (Independent Power Producers), project developers, and utility-scale projects. The purpose is to evaluate opportunities from the {companies_list} for Hillingdale Capital to approach and connect with relevant parties.",
    verbose=True
)

In [10]:
research = Task(
    description = (
        "1. Research news in the Africa and Europe renewable energy market and collate a list of companies that require project or growth financing"
    ),
    expected_output="A list of company names that require financing into a companies_list that will be parsed to the evaluator agent",
    agent=researcher
)

In [12]:
evaluate = Task(
    description = (
        "1. Evaluate the {companies_list} and produce a .csv file with company name, score (of how needy they are for financing), and reasons for their score"
    ),
    expected_output=".csv file with company name, score and reason for score",
    agent=evaluator
)

In [13]:
crewman = Crew(
    agents=[researcher, evaluator],
    tasks=[research, evaluate],
    verbose=True
)

In [14]:
result = crewman.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 62180264-0d16-4989-a092-929d9281edd6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Research news in the Africa and Europe renewable energy market and collate a list of companies that   │
│  require project or growth financing                                                                            │
│  ID: 6db8a9d1-faff-433a-a182-9f5d8c52ccbe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: 1. Research news in the Africa and Europe renewable energy market and collate a list of companies that   │
│  require project or growth financing                                                                            │
│  Agent: researcher                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 62180264-0d16-4989-a092-929d9281edd6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RuntimeError: Agent execution was invoked synchronously from within a running event loop. Use `agent.kickoff_async()` / `crew.kickoff_async()` (or `await agent.aexecute_task(...)`) when calling from async code.